# PitWall - QLoRA Fine-tuning for Qwen2.5-3B-Instruct

**Platform:** Kaggle T4 16GB  
**Base model:** Qwen2.5-3B-Instruct  
**Training method:** 4-bit QLoRA with Unsloth  

This notebook is the Qwen version of your earlier Llama workflow, cleaned up for Kaggle and for local offline use afterward.

**Before running:**
1. Add `HF_TOKEN` in Kaggle Secrets if you want to push the adapter to Hugging Face.
2. Attach your dataset input containing `train.jsonl` and `val.jsonl`.
3. Edit `HF_REPO`, `TRAIN_PATH`, and `VAL_PATH` in the config cell.

**Important local offline note:**
A QLoRA adapter cannot run by itself. For fully local offline inference on your laptop, you need:
- the base model weights for `Qwen/Qwen2.5-3B-Instruct` stored locally, and
- the fine-tuned adapter stored locally.

That still fits your goal: after one-time download, the app can run entirely offline from local files.

In [ ]:
# Cell 1 - Install dependencies
# Run this cell first, then restart the kernel before continuing.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--quiet",
        "unsloth",
        "transformers>=4.45.0",
        "datasets",
        "trl",
        "peft",
        "accelerate",
        "bitsandbytes",
        "huggingface_hub",
    ],
    check=True,
)

print("Install complete. Restart the kernel now, then run the remaining cells.")

In [ ]:
# Cell 2 - Config and secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ["UNSLOTH_USE_CUT_CROSS_ENTROPY"] = "0"

# Edit these paths for your Kaggle input dataset
TRAIN_PATH = "/kaggle/input/pitwall-training/train.jsonl"
VAL_PATH = "/kaggle/input/pitwall-training/val.jsonl"

# Adapter output directory
OUTPUT_DIR = "/kaggle/working/pitwall-qwen25-3b-adapter"

# Optional Hugging Face repo for adapter upload
HF_REPO = "mehuld27/pitwall-qwen25-3b-adapter"

# Qwen base model for Unsloth 4-bit loading
BASE_MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
BASE_MODEL_ORIGINAL = "Qwen/Qwen2.5-3B-Instruct"

# Training settings
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
PER_DEVICE_BATCH = 4
GRAD_ACCUM = 4
SEED = 42

user_secrets = UserSecretsClient()
try:
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN found.")
except Exception:
    HF_TOKEN = None
    print("HF_TOKEN not found. Training is fine; push-to-hub cells will be skipped.")

print("BASE_MODEL_ID:", BASE_MODEL_ID)
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:", VAL_PATH)

In [ ]:
# Cell 3 - Imports
import json
import logging
from pathlib import Path

from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pitwall_qwen")

print("Imports OK.")

In [ ]:
# Cell 4 - Load Qwen model and tokenizer in 4-bit
log.info("Loading %s in 4-bit", BASE_MODEL_ID)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print("Model and tokenizer loaded.")

In [ ]:
# Cell 5 - Apply QLoRA adapters
log.info("Applying QLoRA adapters")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("QLoRA adapters applied.")

In [ ]:
# Cell 6 - Load JSONL and apply chat template
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def apply_chat_template_batch(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}


log.info("Loading train and val datasets")
train_raw = load_jsonl(TRAIN_PATH)
val_raw = load_jsonl(VAL_PATH)

train_dataset = Dataset.from_list(train_raw).map(
    apply_chat_template_batch,
    batched=True,
    remove_columns=["messages"],
)
val_dataset = Dataset.from_list(val_raw).map(
    apply_chat_template_batch,
    batched=True,
    remove_columns=["messages"],
)

print(f"Train examples: {len(train_dataset)}")
print(f"Val examples:   {len(val_dataset)}")
print(train_dataset[0]["text"][:800])

In [ ]:
# Cell 7 - Configure trainer
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
    dataset_num_proc=1,
    packing=False,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("Trainer configured.")

In [ ]:
# Cell 8 - Train
log.info("Starting training")
log.info("Base model: %s", BASE_MODEL_ID)
log.info("Epochs=%d, batch=%d, grad_accum=%d, lr=%s", NUM_EPOCHS, PER_DEVICE_BATCH, GRAD_ACCUM, LEARNING_RATE)

train_result = trainer.train()
log.info("Training complete")
train_result

In [ ]:
# Cell 9 - Save adapter locally
log.info("Saving adapter to %s", OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

In [ ]:
# Cell 10 - Quick smoke test inside Kaggle
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an F1 race engineer with deep knowledge of tyre behaviour, race strategy, driving mechanics, and circuit characteristics across the 2022-2025 seasons. Give specific, data-grounded answers. Never guess -- if you don't know, say so."},
    {"role": "user", "content": "Soft tyres are 12 laps old, degradation is 0.085 s/lap, cliff is lap 15, and gap behind is 2.5 seconds. Should we stay out or pit?"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
outputs = model.generate(inputs, max_new_tokens=180, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Cell 11 - Optional: push adapter to Hugging Face
if HF_TOKEN is None:
    print("HF_TOKEN missing, skipping push.")
else:
    log.info("Pushing adapter to %s", HF_REPO)
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Adapter pushed to {HF_REPO}")

## Running PitWall fully offline on your laptop

### Option A: local Hugging Face + PEFT adapter
This is the cleanest way to use the QLoRA adapter directly.

You need both stored locally:
- base model: `Qwen/Qwen2.5-3B-Instruct`
- adapter folder: your saved `pitwall-qwen25-3b-adapter`

Typical local loading pattern:

```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model_path = r"C:\\models\\Qwen2.5-3B-Instruct"
adapter_path = r"C:\\models\\pitwall-qwen25-3b-adapter"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_path,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, adapter_path)
FastLanguageModel.for_inference(model)
```

This works offline after the base model and adapter are already on disk.

### Option B: merge/export later for Ollama or GGUF
If you want to keep your current Flask app unchanged, the easiest production path is usually:
- train adapter in Kaggle,
- merge/export separately,
- load the resulting local model in Ollama.

That is simpler operationally, but it is not adapter-only inference.

### Best fit for your project
Because you want Qwen plus local offline usage in Flask, the most faithful route is Option A: load base model + adapter locally in Python and update the Flask inference path to use Transformers/PEFT instead of Ollama.